In [25]:
# Cell 1: Imports and Setup
import os, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import re



In [26]:
# Cell 2: Random Seed Setup for Reproducibility  
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Cell 3: Device Configuration (GPU/CPU/MPS)
device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: cuda


In [27]:
# Cell 4: Training Hyperparameters
n_epochs = 5
learning_rate = 5e-6  
batch_size = 4

# Cell 5: Constants and Label Definitions
# Define polarity labels
POLARITY_LABELS = ['positive', 'negative', 'neutral']  

# Cell 6: Directory Creation for Models and Results
#MODEL_DIR = "model"
#os.makedirs(MODEL_DIR, exist_ok=True)  # Create the base model directory
#os.makedirs("results", exist_ok=True)

In [28]:
# Cell 7: Text Preprocessing Function

# Download stopwords if not already present
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    """
    Preprocess text for biomedical NLP by cleaning and normalizing
    
    Args:
        text: Raw text string to preprocess
        
    Returns:
        Cleaned and normalized text string
    """
    # Handle NaN values
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Keep hyphens as they may be important in biomedical terms (e.g., auto-regulation)
    text = re.sub(r'[^\w\s-]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove stopwords but keep important biomedical terms
    # Note: We're being conservative with stopword removal for biomedical text
    text = " ".join([word.strip() for word in text.split() if word not in stop_words or len(word) > 4])
    
    return text.strip()


# Cell 8: Dataset Class Definition  
class PubMedDataset(Dataset):
    """
    Dataset class for PubMed text classification with optional polarity labels
    
    This class handles both mechanism detection and polarity classification tasks
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=512, polarities=None):
        """
        Initialize the dataset
        
        Args:
            texts: List or Series of text samples
            labels: numpy array of multi-label binary labels for mechanisms
            tokenizer: Transformers tokenizer for text encoding
            max_length: Maximum sequence length for tokenization
            polarities: Optional numpy array of polarity labels (for multi-task learning)
        """
        self.texts = texts.reset_index(drop=True) if hasattr(texts, 'reset_index') else texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.polarities = polarities
        
        # Validate inputs
        if len(self.texts) != len(self.labels):
            raise ValueError(f"Text count ({len(self.texts)}) doesn't match label count ({len(self.labels)})")
        
        if self.polarities is not None and len(self.texts) != len(self.polarities):
            raise ValueError(f"Text count ({len(self.texts)}) doesn't match polarity count ({len(self.polarities)})")
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        """
        Get a single item from the dataset
        
        Args:
            idx: Index of the item to retrieve
            
        Returns:
            Dictionary containing tokenized inputs and labels
        """
        # Get text and preprocess it
        text = str(self.texts.iloc[idx] if hasattr(self.texts, 'iloc') else self.texts[idx])
        text = preprocess_text(text)  # Apply preprocessing
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Prepare the return dictionary
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(self.labels[idx])  # Convert to float for BCE loss
        }
        
        # Add polarity labels if available
        if self.polarities is not None:
            item['polarity'] = torch.LongTensor([self.polarities[idx]]).squeeze()  # Convert to long for CE loss
        
        return item

In [29]:
# Cell 9: Loss Function Classes
class CombinedLoss(nn.Module):
    def __init__(self, pos_weights, gamma=0.5, alpha=0.8):
        super(CombinedLoss, self).__init__()
        self.pos_weights = pos_weights
        self.gamma = gamma
        self.alpha = alpha  # Higher alpha = more weight on BCE
        
    def forward(self, inputs, targets):
        # Weighted BCE loss
        BCE_loss = F.binary_cross_entropy_with_logits(
            inputs, targets, pos_weight=self.pos_weights, reduction='none'
        )
        
        # Focal component (lighter weight)
        pt = torch.exp(-BCE_loss)
        focal_component = (1 - pt) ** self.gamma * BCE_loss
        
        # Combine both losses (80% BCE, 20% Focal)
        combined = self.alpha * BCE_loss + (1 - self.alpha) * focal_component
        
        return combined.mean()
    
class MultiTaskLoss(nn.Module):
    def __init__(self, pos_weights, mech_weight=0.7, pol_weight=0.3):
        super(MultiTaskLoss, self).__init__()
        self.pos_weights = pos_weights
        self.mech_weight = mech_weight
        self.pol_weight = pol_weight
        self.mech_criterion = CombinedLoss(pos_weights)
        self.pol_criterion = nn.CrossEntropyLoss()
        
    def forward(self, mech_logits, pol_logits, mech_labels, pol_labels):
        mech_loss = self.mech_criterion(mech_logits, mech_labels)
        pol_loss = self.pol_criterion(pol_logits, pol_labels)
        
        # Combine mechanism and polarity losses
        total_loss = self.mech_weight * mech_loss + self.pol_weight * pol_loss
        
        return total_loss, mech_loss, pol_loss

## 3. Data Loading and Preprocessing

In [30]:
# Cell 10: Data Loading
# Load data
df = pd.read_csv("/kaggle/input/shuffled-4/train_data.csv")
print(df.shape)
df.head()

(26220, 7)


,AC,PMID,Title,Abstract,Terms,Text_combined,batch_number
0,Q96DE0,15489334,"The status, quality, and expansion of the NIH ...",The National Institutes of Health's Mammalian ...,NaN,"The status, quality, and expansion of the NIH ...",1
1,P41214,16710414,The DNA sequence and biological annotation of ...,The reference sequence for each human chromoso...,NaN,The DNA sequence and biological annotation of ...,1
2,Q03835,14562106,Global analysis of protein expression in yeast.,The availability of complete genomic sequences...,NaN,Global analysis of protein expression in yeast...,1
3,O43719,10913173,Relief of two built-In autoinhibitory mechanis...,Tat stimulation of human immunodeficiency viru...,autoinhibition,Relief of two built-In autoinhibitory mechanis...,1
4,Q9M3D8,15067507,A salt-responsive receptor-like kinase gene re...,NTHK1 is a salt-inducible ethylene receptor ge...,autophosphorylation,A salt-responsive receptor-like kinase gene re...,1


In [31]:
# Cell 11: Text Preprocessing and Cleaning
df['Text_Cleaned'] = df['Text_combined'].apply(preprocess_text)

In [32]:
# Cell 12: Mechanism Label Binarization
# Convert comma-separated terms to multi-label binary format using MLB

# Binarize the Terms column
df['Terms_List'] = df['Terms'].apply(
    lambda x: [term.strip() for term in str(x).split(',')] if pd.notna(x) and x != '' else []
)

# Initialize and fit the MultiLabelBinarizer
mlb = MultiLabelBinarizer()
binary_labels = mlb.fit_transform(df['Terms_List'])

# Get the class names
label_columns = mlb.classes_
print(f"Found {len(label_columns)} unique labels: {label_columns}")

# Create a DataFrame with the binary labels
labels_df = pd.DataFrame(binary_labels, columns=label_columns)

# Save label columns for later use
with open('label_columns.json', 'w') as f:
    json.dump(list(label_columns), f)

# Keep only essential columns
df_cleaned = df[['batch_number', 'Text_Cleaned']].copy()
df_cleaned = pd.concat([df_cleaned, labels_df], axis=1)

print(f"Final cleaned data shape: {df_cleaned.shape}")
print(df_cleaned.shape)
df_cleaned.head()

Found 10 unique labels: ['autoactivation' 'autocatalysis' 'autofeedback' 'autoinduction'
 'autoinhibition' 'autokinase' 'autolysis' 'autophosphorylation'
 'autoregulation' 'autoubiquitination']
Final cleaned data shape: (26220, 12)
(26220, 12)


,batch_number,Text_Cleaned,autoactivation,autocatalysis,autofeedback,autoinduction,autoinhibition,autokinase,autolysis,autophosphorylation,autoregulation,autoubiquitination
0,1,status quality expansion nih full-length cdna ...,0,0,0,0,0,0,0,0,0,0
1,1,dna sequence biological annotation human chrom...,0,0,0,0,0,0,0,0,0,0
2,1,global analysis protein expression yeast avail...,0,0,0,0,0,0,0,0,0,0
3,1,relief two built-in autoinhibitory mechanisms ...,0,0,0,0,1,0,0,0,0,0
4,1,salt-responsive receptor-like kinase gene regu...,0,0,0,0,0,0,0,1,0,0


In [33]:
# Cell 13: Polarity Label Inference and Encoding
# Add polarity inference function
def infer_polarity(text, mechanism):
    """
    Infer polarity (positive/negative/neutral) from text and mechanism
    
    This is a rule-based method that can be later replaced with manual annotations
    """
    # EXPANDED: Keywords indicating negative regulation
    negative_keywords = [
    # Already included
    'inhibit', 'repress', 'suppress', 'block', 'reduce', 'decrease', 'down-regulat', 
    'downregulat', 'negative', 'inactivat', 'stop', 'prevent', 'attenuate', 'dampen',
    'silence', 'knock', 'impair', 'abolish', 'diminish', 'weaken', 'curtail', 'halt', 
    'terminate', 'cease', 'limit', 'restrict', 'degradation', 'breakdown', 'turnover', 
    'cleavage', 'proteolysis', 'downmodulat', 'counter', 'antagoniz', 'oppose',

    # New additions
    'desensitiz', 'depress', 'disrupt', 'disconnect', 'decay', 'decouple',
    'detachment', 'abrogate', 'retract', 'withdraw', 'suspend', 'neutralize',
    'disassemble', 'inhibition', 'negatively', 'interfere', 'destabiliz',
    'decimate', 'destroy', 'nullify', 'quench', 'obstruct', 'restrain'
]

    
    # EXPANDED: Keywords indicating positive regulation  
    positive_keywords = [
    # Already included
    'activat', 'increas', 'induce', 'enhance', 'promot', 'stimulat', 'up-regulat',
    'upregulat', 'positive', 'amplif', 'boost', 'augment', 'facilitate', 'accelerate',
    'catalyze', 'drive', 'trigger', 'elicit', 'evoke', 'potentiat', 'strengthen',
    'reinforce', 'foster', 'support', 'maintain', 'sustain', 'stabiliz', 'preserve',
    'accumul', 'recruit', 'upmodulat', 'elevat', 'heighten', 'agoniz',

    # New additions
    'reactivat', 'intensify', 'initiate', 'commit', 'enhancement', 'potency',
    'enable', 'reinitiate', 'facilitation', 'recovery', 'perpetuate', 'favor',
    'amplification', 'synergize', 'revive', 'regenerate', 'excite', 'reassemble',
    'elevation', 'reengage', 'encourage', 'rescue', 'drive up'
]

    
    # Convert text to lowercase for matching
    text_lower = text.lower()
    
    # Count keyword occurrences (more robust than just presence)
    negative_count = sum(1 for keyword in negative_keywords if keyword in text_lower)
    positive_count = sum(1 for keyword in positive_keywords if keyword in text_lower)
    
    # Determine polarity based on keyword balance
    if negative_count > positive_count:
        return 'negative'
    elif positive_count > negative_count:
        return 'positive'
    else:
        # If balanced or no keywords, use mechanism-based heuristics
        if mechanism in ['autoinhibition', 'autorepression']:  # Typically negative
            return 'negative'
        elif mechanism in ['autoactivation', 'autophosphorylation', 'autoinduction']:  # Typically positive
            return 'positive'
        else:
            return 'neutral'  # Default
        
# Add polarity labels to dataset
print("Inferring polarity labels...")
polarities = []

for idx, row in df.iterrows():
    text = row['Text_combined'] if pd.notna(row['Text_combined']) else ""
    
    # Get the mechanisms for this example
    mechanisms = row['Terms_List']
    
    # If no mechanisms, assign neutral
    if not mechanisms:
        polarities.append('neutral')
    else:
        # Get the first mechanism (for multi-labeled entries)
        mechanism = mechanisms[0] if mechanisms else ""
        polarity = infer_polarity(text, mechanism)
        polarities.append(polarity)

# Add to dataframe
df['polarity'] = polarities

# Encode polarity labels
polarity_encoder = LabelEncoder()
polarity_encoder.fit(POLARITY_LABELS)  # Use our predefined labels
encoded_polarities = polarity_encoder.transform(df['polarity'])

# Add to cleaned dataframe
df_cleaned['polarity'] = df['polarity']
df_cleaned['polarity_encoded'] = encoded_polarities

# Save polarity encoder classes
with open('polarity_labels.json', 'w') as f:
    json.dump(list(polarity_encoder.classes_), f)

# Display polarity distribution
polarity_counts = df['polarity'].value_counts()
print("\nPolarity distribution:")
for pol, count in polarity_counts.items():
    print(f"{pol}: {count} ({count/len(df)*100:.1f}%)")


Inferring polarity labels...

Polarity distribution:
neutral: 18120 (69.1%)
positive: 5015 (19.1%)
negative: 3085 (11.8%)


In [34]:
# Cell 14: Final Dataset 
print(f"Final cleaned data shape with polarity: {df_cleaned.shape}")
df_cleaned.head()

Final cleaned data shape with polarity: (26220, 14)


,batch_number,Text_Cleaned,autoactivation,autocatalysis,autofeedback,autoinduction,autoinhibition,autokinase,autolysis,autophosphorylation,autoregulation,autoubiquitination,polarity,polarity_encoded
0,1,status quality expansion nih full-length cdna ...,0,0,0,0,0,0,0,0,0,0,neutral,1
1,1,dna sequence biological annotation human chrom...,0,0,0,0,0,0,0,0,0,0,neutral,1
2,1,global analysis protein expression yeast avail...,0,0,0,0,0,0,0,0,0,0,neutral,1
3,1,relief two built-in autoinhibitory mechanisms ...,0,0,0,0,1,0,0,0,0,0,positive,2
4,1,salt-responsive receptor-like kinase gene regu...,0,0,0,0,0,0,0,1,0,0,negative,0


## Model Architecture

In [35]:
class LabelWiseAttention(nn.Module):
    def __init__(self, hidden_size, num_labels):
        super(LabelWiseAttention, self).__init__()
        self.label_query = nn.Parameter(torch.randn(num_labels, hidden_size))
        self.linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, encoder_output, attention_mask):
        Q = self.label_query
        K = self.linear(encoder_output)
        attention_scores = torch.matmul(K, Q.t())
        attention_scores = attention_scores.masked_fill(
            attention_mask.unsqueeze(-1) == 0, float('-inf')
        )
        attention_weights = torch.softmax(attention_scores, dim=1)
        weighted_sum = torch.einsum("bsl,bsh->blh", attention_weights, encoder_output)
        return weighted_sum


class LabelWiseAttentionBinaryPolarityClassifier(nn.Module):
    def __init__(self, model_name, n_mech_labels):
        super(LabelWiseAttentionBinaryPolarityClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Label-wise attention mechanism head
        self.attention = LabelWiseAttention(hidden_size, n_mech_labels)
        self.mech_classifier = nn.Linear(hidden_size, 1)  # one per label

        # 3-class polarity head (positive vs. negative vs. neutral)
        self.polarity_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 3)  # 3 classes
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        encoder_output = outputs.last_hidden_state

        # Mechanism predictions
        attended = self.attention(encoder_output, attention_mask)  # [B, L, H]
        mech_logits = self.mech_classifier(attended).squeeze(-1)   # [B, L]

        # Polarity prediction (use CLS token)
        cls_output = encoder_output[:, 0, :]
        polarity_logits = self.polarity_head(cls_output)    # [B, 3]

        return mech_logits, polarity_logits


### I delete cell 16. You only need one model class to perform multi-task learning. The new class does everything PolarityPubMedBERTClassifier does, but better:It uses label-wise attention for mechanism prediction.It simplifies polarity to binary (positive vs. negative).



## Prepare Inputs for DataLoader

In [36]:
#Step 1: Split Dataset
X = df_cleaned["Text_Cleaned"]
y = df_cleaned[label_columns].values
polarity = df_cleaned["polarity_encoded"].values

# Split into training and validation sets
X_train, X_val, y_train, y_val, polarity_train, polarity_val = train_test_split(
    X, y, polarity, test_size=0.2, random_state=42
)

In [37]:
#Step 2: Load Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")


In [38]:
#Step 3: Create DataLoaders 

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import torch

# ✅ 1. Weighted sampler for mechanism label balancing
def create_moderate_sampler(y):
    class_sample_count = np.sum(y, axis=0)
    weight_per_class = 1.0 / np.sqrt(np.clip(class_sample_count, 5, np.inf))
    
    sample_weights = np.zeros(len(y))
    for i in range(len(y)):
        if np.sum(y[i]) > 0:
            positive_indices = np.where(y[i] == 1)[0]
            sample_weights[i] = np.mean(weight_per_class[positive_indices])
        else:
            sample_weights[i] = 0.5 / max(1, (len(y) - np.sum(np.any(y, axis=1))))
    
    return WeightedRandomSampler(torch.FloatTensor(sample_weights), len(sample_weights))

# ✅ 2. Dataset class for multi-task model
class PubMedDataset(Dataset):
    def __init__(self, texts, mech_labels, tokenizer, polarities=None, max_length=512):
        self.texts = list(texts)
        self.mech_labels = mech_labels
        self.polarities = polarities
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'mech_labels': torch.FloatTensor(self.mech_labels[idx])
        }
        if self.polarities is not None:
            item['polarity'] = torch.FloatTensor([self.polarities[idx]])  # shape [1]
        return item

# ✅ 3. DataLoader creation function
def create_dataset_and_loader(X, y, batch_size, tokenizer, train=True, polarities=None):
    dataset = PubMedDataset(X, y, tokenizer, polarities=polarities)
    
    if train:
        sampler = create_moderate_sampler(y)
        loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    return loader


In [39]:
train_loader = create_dataset_and_loader(
    X=X_train,
    y=y_train,
    batch_size=16,
    tokenizer=tokenizer,
    train=True,
    polarities=polarity_train
)

# --- Validation loader ---
val_loader = create_dataset_and_loader(
    X=X_val,
    y=y_val,
    batch_size=16,
    tokenizer=tokenizer,
    train=False,  # important: no sampler for val
    polarities=polarity_val
)


model = LabelWiseAttentionBinaryPolarityClassifier(
    model_name="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    n_mech_labels=len(label_columns)
)
model.to(device)



LabelWiseAttentionBinaryPolarityClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((

## Dynamic Threshold

In [40]:
from sklearn.metrics import precision_recall_curve

def compute_dynamic_thresholds(model, data_loader):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["mech_labels"].to(device)

            mech_logits, _ = model(input_ids, attention_mask)
            probs = torch.sigmoid(mech_logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    new_thresholds = []
    for i in range(all_labels.shape[1]):
        precision, recall, thres = precision_recall_curve(all_labels[:, i], all_probs[:, i])
        f1 = 2 * precision * recall / (precision + recall + 1e-6)
        best_idx = np.argmax(f1)
        best_threshold = float(np.clip(thres[best_idx], 0.1, 0.9)) if len(thres) > 0 else 0.5
        new_thresholds.append(best_threshold)

    print("\n📏 Dynamic Thresholds:", [f"{t:.2f}" for t in new_thresholds])
    return new_thresholds



## Evaluation

In [41]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)

def evaluate(model, data_loader, pos_weights, thresholds):
    """
    Evaluate the model's mechanism and polarity performance.
    """
    model.eval()
    total_loss = 0

    all_mech_preds = []
    all_mech_labels = []
    all_pol_preds = []
    all_pol_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            mech_labels = batch['mech_labels'].to(device)
            polarity_labels = batch['polarity'].to(device).squeeze()

            mech_logits, polarity_logit = model(input_ids, attention_mask)

            # Losses
            mech_loss = F.binary_cross_entropy_with_logits(
                mech_logits, mech_labels, pos_weight=pos_weights.to(device)
            )
            pol_loss = F.binary_cross_entropy_with_logits(
                polarity_logit, polarity_labels.float()
            )
            total_loss += 0.7 * mech_loss.item() + 0.3 * pol_loss.item()

            # Predictions
            mech_probs = torch.sigmoid(mech_logits).cpu().numpy()
            polarity_probs = torch.sigmoid(polarity_logit).cpu().numpy()

            # Threshold-based mech prediction
            mech_pred = np.array([
                (mech_probs[:, i] >= thresholds[i]).astype(int)
                for i in range(len(thresholds))
            ]).T

            pol_pred = (polarity_probs >= 0.5).astype(int)

            # Store
            all_mech_preds.extend(mech_pred)
            all_mech_labels.extend(mech_labels.cpu().numpy())
            all_pol_preds.extend(pol_pred)
            all_pol_labels.extend(polarity_labels.cpu().numpy())

    # Convert to arrays
    all_mech_preds = np.array(all_mech_preds)
    all_mech_labels = np.array(all_mech_labels)
    all_pol_preds = np.array(all_pol_preds)
    all_pol_labels = np.array(all_pol_labels)

    # Mechanism Metrics
    samples_precision = precision_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)
    samples_recall = recall_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)
    samples_f1 = f1_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)
    micro_f1 = f1_score(all_mech_labels, all_mech_preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_mech_labels, all_mech_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_mech_labels, all_mech_preds, average='weighted', zero_division=0)

    # Polarity Metrics
    pol_acc = accuracy_score(all_pol_labels, all_pol_preds)
    pol_f1 = f1_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)
    pol_prec = precision_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)
    pol_rec = recall_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)


    # Final Output
    avg_loss = total_loss / len(data_loader)

    print(f"\n📊 MECHANISM TASK")
    print(f"Loss: {avg_loss:.4f} | Micro F1: {micro_f1:.4f} | Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f} | Samples F1: {samples_f1:.4f}")
    print(f"Samples Precision: {samples_precision:.4f} | Samples Recall: {samples_recall:.4f}")
    
    print(f"\n⚡ POLARITY TASK")
    print(f"Accuracy: {pol_acc:.4f} | F1: {pol_f1:.4f} | Precision: {pol_prec:.4f} | Recall: {pol_rec:.4f}")

    return {
        'loss': avg_loss,
        'mech_f1_micro': micro_f1,
        'mech_f1_macro': macro_f1,
        'mech_f1_weighted': weighted_f1,
        'mech_f1_samples': samples_f1,
        'mech_precision_samples': samples_precision,
        'mech_recall_samples': samples_recall,
        'polarity_acc': pol_acc,
        'polarity_f1': pol_f1,
        'polarity_precision': pol_prec,
        'polarity_recall': pol_rec
    }


## Training

In [42]:
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# -------- Setup --------
batch_number = 1  # select which batch to train on
model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

print(f"\n🟦 Processing Batch {batch_number} ...")

# -------- Optimizer & Weights --------
pos_weights = torch.tensor(1.0).repeat(len(label_columns)).to(device)  # placeholder
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

# -------- Dynamic Threshold Computation --------
def set_thresholds(pos_weights):
    thresholds = []
    min_w, max_w = pos_weights.min().item(), pos_weights.max().item()
    w_range = max(max_w - min_w, 1e-6)  # avoid division by zero

    for w in pos_weights:
        norm = (w.item() - min_w) / w_range
        t = 0.8 - 0.6 * norm
        thresholds.append(max(0.2, min(t, 0.8)))

    formatted = [f"{t:.2f}" for t in thresholds]
    print(f"\n📐 Dynamic Thresholds: {formatted}")
    return thresholds

# -------- Training --------
num_epochs = 5
print(f"🔁 Training on FULL batch {batch_number} for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        mech_labels = batch["mech_labels"].to(device)
        polarity_labels = batch["polarity"].to(device).squeeze()

        mech_logits, polarity_logits = model(input_ids, attention_mask)

        mech_loss = F.binary_cross_entropy_with_logits(mech_logits, mech_labels, pos_weight=pos_weights)
        pol_loss = F.cross_entropy(polarity_logits, polarity_labels.long())
        total_loss = 0.7 * mech_loss + 0.3 * pol_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item()
        print(f"🐣 Epoch {epoch + 1}/{num_epochs} | Batch {batch_idx + 1} - Loss: {total_loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    print(f"✅ Epoch {epoch + 1} complete | Avg Loss: {avg_loss:.4f}")

# -------- Compute Dynamic Thresholds --------
thresholds = set_thresholds(pos_weights)

# -------- Evaluation --------
def evaluate(model, data_loader, pos_weights, thresholds):
    model.eval()
    total_loss = 0
    all_mech_preds, all_mech_labels = [], []
    all_pol_preds, all_pol_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            mech_labels = batch["mech_labels"].to(device)
            polarity_labels = batch["polarity"].to(device).squeeze()

            mech_logits, polarity_logits = model(input_ids, attention_mask)

            mech_loss = F.binary_cross_entropy_with_logits(mech_logits, mech_labels, pos_weight=pos_weights)
            pol_loss = F.cross_entropy(polarity_logits, polarity_labels.long())
            loss = 0.7 * mech_loss + 0.3 * pol_loss
            total_loss += loss.item()

            mech_probs = torch.sigmoid(mech_logits).cpu().numpy()
            mech_pred = np.array([
                (mech_probs[:, i] >= thresholds[i]).astype(int)
                for i in range(len(thresholds))
            ]).T

            pol_pred = torch.argmax(polarity_logits, dim=1).cpu().numpy()

            all_mech_preds.extend(mech_pred)
            all_mech_labels.extend(mech_labels.cpu().numpy())
            all_pol_preds.extend(pol_pred)
            all_pol_labels.extend(polarity_labels.cpu().numpy())

    # Metrics
    all_mech_preds, all_mech_labels = np.array(all_mech_preds), np.array(all_mech_labels)
    all_pol_preds, all_pol_labels = np.array(all_pol_preds), np.array(all_pol_labels)

    # Mechanism
    samples_f1 = f1_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)
    micro_f1 = f1_score(all_mech_labels, all_mech_preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_mech_labels, all_mech_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_mech_labels, all_mech_preds, average='weighted', zero_division=0)
    samples_precision = precision_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)
    samples_recall = recall_score(all_mech_labels, all_mech_preds, average='samples', zero_division=0)

    # Polarity
    pol_acc = accuracy_score(all_pol_labels, all_pol_preds)
    pol_f1 = f1_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)
    pol_prec = precision_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)
    pol_rec = recall_score(all_pol_labels, all_pol_preds, average='macro', zero_division=0)

    # Print
    print(f"\n📊 MECHANISM TASK")
    print(f"Loss: {total_loss / len(data_loader):.4f} | Micro F1: {micro_f1:.4f} | Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f} | Samples F1: {samples_f1:.4f}")
    print(f"Samples Precision: {samples_precision:.4f} | Samples Recall: {samples_recall:.4f}")
    print(f"\n⚡ POLARITY TASK (3-class)")
    print(f"Accuracy: {pol_acc:.4f} | F1 (macro): {pol_f1:.4f} | Precision: {pol_prec:.4f} | Recall: {pol_rec:.4f}")

    return {
        'loss': total_loss / len(data_loader),
        'mech_f1_micro': micro_f1,
        'mech_f1_macro': macro_f1,
        'mech_f1_weighted': weighted_f1,
        'mech_f1_samples': samples_f1,
        'mech_precision_samples': samples_precision,
        'mech_recall_samples': samples_recall,
        'polarity_acc': pol_acc,
        'polarity_f1': pol_f1,
        'polarity_precision': pol_prec,
        'polarity_recall': pol_rec
    }

# -------- Run evaluation --------
print("\n📊 Evaluating on validation set...")
val_metrics = evaluate(model, val_loader, pos_weights, thresholds)

# -------- Save everything --------
model_dir = "/kaggle/working"
os.makedirs(model_dir, exist_ok=True)

torch.save(model.state_dict(), f"{model_dir}/test_model_batch_{batch_number}.pt")
print(f"✅ Model saved to: {model_dir}/test_model_batch_{batch_number}.pt")

with open(f"{model_dir}/test_thresholds_batch_{batch_number}.json", "w") as f:
    json.dump(thresholds, f)
print(f"✅ Thresholds saved to: {model_dir}/test_thresholds_batch_{batch_number}.json")

with open(f"{model_dir}/test_metrics_batch_{batch_number}.json", "w") as f:
    json.dump(val_metrics, f, indent=2)
print(f"✅ Metrics saved to: {model_dir}/test_metrics_batch_{batch_number}.json")




🟦 Processing Batch 1 ...
🔁 Training on FULL batch 1 for 5 epochs...
🐣 Epoch 1/5 | Batch 1 - Loss: 0.7473
🐣 Epoch 1/5 | Batch 2 - Loss: 0.7375
🐣 Epoch 1/5 | Batch 3 - Loss: 0.6993
🐣 Epoch 1/5 | Batch 4 - Loss: 0.6924
🐣 Epoch 1/5 | Batch 5 - Loss: 0.6559
🐣 Epoch 1/5 | Batch 6 - Loss: 0.6424
🐣 Epoch 1/5 | Batch 7 - Loss: 0.6384
🐣 Epoch 1/5 | Batch 8 - Loss: 0.6245
🐣 Epoch 1/5 | Batch 9 - Loss: 0.6124
🐣 Epoch 1/5 | Batch 10 - Loss: 0.5842
🐣 Epoch 1/5 | Batch 11 - Loss: 0.5865
🐣 Epoch 1/5 | Batch 12 - Loss: 0.5735
🐣 Epoch 1/5 | Batch 13 - Loss: 0.5762
🐣 Epoch 1/5 | Batch 14 - Loss: 0.5764
🐣 Epoch 1/5 | Batch 15 - Loss: 0.5617
🐣 Epoch 1/5 | Batch 16 - Loss: 0.5620
🐣 Epoch 1/5 | Batch 17 - Loss: 0.5645
🐣 Epoch 1/5 | Batch 18 - Loss: 0.5538
🐣 Epoch 1/5 | Batch 19 - Loss: 0.5382
🐣 Epoch 1/5 | Batch 20 - Loss: 0.5595
🐣 Epoch 1/5 | Batch 21 - Loss: 0.5573
🐣 Epoch 1/5 | Batch 22 - Loss: 0.5452
🐣 Epoch 1/5 | Batch 23 - Loss: 0.5526
🐣 Epoch 1/5 | Batch 24 - Loss: 0.5315
🐣 Epoch 1/5 | Batch 25 - Los

In [43]:
print("Batch numbers in dataset:", df_cleaned["batch_number"].unique())


Batch numbers in dataset: [1 2 3 4 5]


In [44]:
batch_df = df_cleaned[df_cleaned["batch_number"] == batch_number]
print(batch_df.shape)  # should NOT be (0, ...)


(5244, 14)


## Testing 


In [2]:
from transformers import AutoTokenizer
import torch

# Re-import model class if needed
model = LabelWiseAttentionBinaryPolarityClassifier(
    model_name="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    n_mech_labels=len(label_columns)
)

# Load weights
model.load_state_dict(torch.load("/kaggle/input/output/test_model_batch_1.pt"))
model.to(device)
model.eval()


for text, true_mech, true_pol in implicit_examples:
    # Tokenize
    encoding = tokenizer(text, padding='max_length', truncation=True, max_length=256, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Model forward
    with torch.no_grad():
        mech_logits, pol_logits = model(input_ids, attention_mask)

    # Prediction: Mechanism
    mech_probs = torch.sigmoid(mech_logits).squeeze(0).cpu().numpy()
    mech_preds = [mlb.classes_[i] for i, p in enumerate(mech_probs) if p >= thresholds[i]]

    # Prediction: Polarity
    pol_pred_idx = torch.argmax(pol_logits, dim=1).item()
    pol_pred_label = polarity_encoder.classes_[pol_pred_idx]

    print("📘 TEXT:", text)
    print("🔹 True Mechanism:", true_mech)
    print("🔸 Predicted Mechanisms:", mech_preds)
    print("🔹 True Polarity:", true_pol)
    print("🔸 Predicted Polarity:", pol_pred_label)
    print("------")


NameError: name 'LabelWiseAttentionBinaryPolarityClassifier' is not defined

In [ ]:
 # FIXED: Create examples with BOTH mechanism and polarity labels
implicit_examples = [
        # Format: (text, mechanism_label, polarity_label)
        ("The transcription factor binds to its own promoter region.", "autoregulation", "neutral"),
        ("The enzyme activates itself through conformational change.", "autoactivation", "positive"),
        ("The protein phosphorylates itself on a tyrosine residue.", "autophosphorylation", "positive"),
        ("The protease cleaves itself to generate the active form.", "autocatalysis", "positive"),
        ("The cell produces molecules that signal itself to change behavior.", "autoinduction", "positive"),
        ("The receptor signals to reduce its own expression level.", "autoinhibition", "negative"),
        ("Upon binding ligand, the receptor undergoes a conformational change that enables phosphorylation of its cytoplasmic domain.", "autophosphorylation", "positive"),
        ("The transcription factor negatively controls expression of its own gene.", "autoregulation", "negative"),
        ("The kinase domain transfers phosphate groups to residues within the same protein.", "autophosphorylation", "positive"),
        ("This bacterial system uses cell-to-cell signaling to coordinate population behavior.", "autoinduction", "positive"),
        ("The peptide recognizes and binds specifically to the same protein it was derived from.", "autofeedback", "neutral"),
        ("The dimeric protein activates by cross-phosphorylation between the two identical subunits.", "autoactivation", "positive"),
        ("AGPCRs uniquely contain large, self-proteolyzing extracellular regions.", "autocatalysis", "positive"),
        ("GAIN domain-mediated self-cleavage is constitutive and produces two-fragment holoreceptors.", "autocatalysis", "positive"),
        ("The self-repression function of IbpA is conserved in other γ-proteobacterial IbpAs.", "autoinhibition", "negative"),
        ("A cationic residue-rich region is critical for the self-suppression activity.", "autoinhibition", "negative"),
        ("We propose a negative feedback loop, in which sphingosine inhibits GBA2 activity.", "autoinhibition", "negative"),
        ("DNA damage-induced activation of p53 initiates a negative-feedback loop which rapidly downregulates RAG1 levels.", "autoregulation", "negative")
    ]

In [ ]:
import os
print(os.listdir("/kaggle/working"))
